In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallExceedance"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"#; spinup_hours="-16"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#DATA LOADING

In [ ]:
# #Loading Radar Mask #decided not to use here
# RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData_NSSL) 

In [ ]:
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

In [ ]:
#DATA LOADING FOR MRMS QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates
    
def GetMRMS_QPE_DataDirectory(ModelData, product="MultiSensor_QPE_01H_Pass2_00.00"):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates2[0]}_{simulationDates2[-1]}",product)
    return inputDirectory
    
def ReadPrecipData_MRMS_QPE(ModelData, yearmonthdayhour, inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]

    MRMS_region = "CONUS" if ModelData.region=="TRACER" else "HAWAII"
    inputPath = os.path.join(inputDirectory,f"MRMSQPE_{MRMS_region}_{ModelData.region}_{yearmonthday}-{hour}0000.nc")
    try:
        precipData = xr.open_dataset(inputPath)["MultiSensor_QPE_01H_Pass2_00.00"].isel(time=0)
    except:
        print(f"{inputPath} does not exist in MRMS data ==> skipping")
        precipData = None
    #Note: data is in mm units
    return precipData

In [ ]:
###################
#FUNCTIONS

In [ ]:
#Calculation Functions

def GetModelWindowTotals(ModelData, targetLat, targetLon, window_hours=3, start_hour=12, end_hour=None):

    """
    Returns array (nwindows, nlat, nlon) of rainfall totals (mm) over non-overlapping
    windows, starting at start_hour, interpolated to the target (MRMS) grid.
    """
    start, nHours = GetRunHours(ModelData)
    if end_hour is not None: nHours = end_hour   
    hours = np.arange(start_hour, nHours + 1, window_hours)

    acc = []
    for h in tqdm(hours, desc="Model accumulation"):
        idx, ts = FindIndexAtHourOffset(ModelData.timeStrings, int(h))
        data = ModelData.GetDataTimestep(ts, printout=False)
        a = data['rainnc'] + data['rainc']
        a = DataSubsetting_Class.SubsetDataRegion(a, ModelData)
        a = a.interp(longitude=targetLon, latitude=targetLat)
        acc.append(a.values)

    acc = np.stack(acc)                # (nhours, nlat, nlon), running accumulation
    totals = np.diff(acc, axis=0)      # window k = acc[k+1] - acc[k]
    totals[totals < 0] = 0             # remove tiny negatives from interpolation
    return totals

def GetMRMSWindowTotals(ModelData, window_hours=3, start_hour=12, end_hour=None):
    """
    Returns totals (nwindows, nlat, nlon) in mm, plus the MRMS grid (lat, lon).
    Windows with any missing hourly file are skipped (no partial sums).
    """
    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData)
    start, nHours = GetRunHours(ModelData)
    if end_hour is not None: nHours = end_hour 

    # hourly QPE files are valid at the END of each hour: start+start_hour+1, ..., start+nHours
    hourEnds = [start + timedelta(hours=int(h)) for h in range(start_hour + 1, nHours + 1)]

    hourly = {}
    for t in tqdm(hourEnds, desc="MRMS hourly QPE"):
        da = ReadPrecipData_MRMS_QPE(ModelData, t.strftime("%Y%m%d%H"), inputDirectory)
        if da is None:
            continue
        da = da.assign_coords(longitude=da.longitude - 360).sortby("latitude")
        da = DataSubsetting_Class.SubsetDataRegion(da, ModelData)
        hourly[t] = da.where(da >= 0)          # MRMS uses negative codes for missing/no coverage

    totals = []
    for k in range(0, len(hourEnds) - window_hours + 1, window_hours):
        chunk = hourEnds[k:k + window_hours]
        if all(t in hourly for t in chunk):
            totals.append(sum(hourly[t].values for t in chunk))

    grid = next(iter(hourly.values()))
    return np.stack(totals), grid.latitude, grid.longitude

def GetRunHours(ModelData):
    """Start time and number of whole hours in the run."""
    start = datetime.strptime(ModelData.timeStrings[0], FMT)
    end   = datetime.strptime(ModelData.timeStrings[-1], FMT)
    return start, int((end - start).total_seconds() // 3600)

def ExceedanceCurve(totals_mm, window_hours, thresholds, wetCutoff=0.1):
    """
    totals_mm : array of window totals (mm), any shape
    Returns (exceed_all, exceed_wet, wetFraction):
      exceed_all : fraction of ALL valid grid-point windows with rate >= threshold
      exceed_wet : same, but conditional on rate >= wetCutoff (mm/h)
      wetFraction: fraction of valid grid-point windows that are wet
    """
    rate = np.asarray(totals_mm).ravel() / window_hours        # mm/h, so 1 h and 3 h share the axis
    rate = rate[np.isfinite(rate)]
    wet = np.sort(rate[rate >= wetCutoff])
    wetFraction = wet.size / rate.size
    if wet.size == 0:
        nan = np.full(len(thresholds), np.nan)
        return nan, nan, wetFraction
    exceed_wet = (wet.size - np.searchsorted(wet, thresholds, side="left")) / wet.size
    return exceed_wet * wetFraction, exceed_wet, wetFraction

def FindIndexAtHourOffset(timeStrings, hourOffset):
    # Convert strings → datetime objects
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in timeStrings]
    
    # Starting time
    start = dt_list[0]
    target = start + timedelta(hours=hourOffset)
    
    # Find closest timestamp
    closest_dt = min(dt_list, key=lambda x: abs(x - target))
    idx = dt_list.index(closest_dt)
    
    return idx, timeStrings[idx]

FMT = "%Y-%m-%d_%H.%M.%S"

In [ ]:
def BuildExceedanceCurves(ModelData_NSSL, ModelData_TEMPO, windows=((3, "-"),),
                          thresholds=None, start_hour=12, end_hour=None,
                          labels=("NSSL", "MRMS QPE", "TEMPO"),
                          colors=("blue", "black", "green")):
    """
    windows    : sequence of (window_hours, linestyle), e.g. ((1, "-"), (3, "--"))
    Returns (curves, wetFractions) for PlotExceedance.
      curves        : list of dicts {"y","color","label","ls"}
      wetFractions  : dict {(window_hours, "NSSL"/"MRMS"/"TEMPO"): fraction}
    """
    if thresholds is None:
        thresholds = np.logspace(-1, 2, 40)

    curves, wetFractions = [], {}
    for window_hours, ls in windows:
        mrms, lat, lon = GetMRMSWindowTotals(ModelData_NSSL, window_hours=window_hours,
                                             start_hour=start_hour, end_hour=end_hour)
        nssl  = GetModelWindowTotals(ModelData_NSSL,  lat, lon, window_hours=window_hours,
                                     start_hour=start_hour, end_hour=end_hour)
        tempo = GetModelWindowTotals(ModelData_TEMPO, lat, lon, window_hours=window_hours,
                                     start_hour=start_hour, end_hour=end_hour)

        # identical windows and grid points for all three
        n = min(len(mrms), len(nssl), len(tempo))
        mrms, nssl, tempo = mrms[:n], nssl[:n], tempo[:n]
        common = np.isfinite(mrms) & np.isfinite(nssl) & np.isfinite(tempo)
        mrms, nssl, tempo = (np.where(common, x, np.nan) for x in (mrms, nssl, tempo))

        suffix = f" {window_hours} h" if len(windows) > 1 else ""   # label windows only if several
        for data, name, label, color in zip((nssl, mrms, tempo),
                                            ("NSSL", "MRMS", "TEMPO"),
                                            labels, colors):
            y, _, wf = ExceedanceCurve(data, window_hours, thresholds)
            curves.append({"y": y, "color": color, "label": label + suffix,
                           "ls": ls, "window_hours": window_hours})
            wetFractions[(window_hours, name)] = wf


    return [curves, wetFractions, thresholds]

def GetExceedanceFilePath(ModelData, windows, start_hour, end_hour):
    wtag = "-".join(str(w) for w, _ in windows)
    fileName = f"ExceedanceCurves_{wtag}h_start{start_hour}_end{end_hour}.pkl"
    return DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager,
                                                outputDirectory, fileName=fileName)

def LoadOrRun_BuildExceedanceCurves(ModelData_NSSL, ModelData_TEMPO, windows=((3, "-"),),
                                    thresholds=None, start_hour=12, end_hour=None,
                                    labels=("NSSL", "MRMS QPE", "TEMPO"),
                                    colors=("blue", "black", "green"),
                                    overwrite=False):
    """
    Loads the saved [curves, wetFractions, thresholds] if the file exists and was made with the
    same settings; otherwise runs BuildExceedanceCurves and saves the result.
    """
    if thresholds is None:
        thresholds = np.logspace(-1, 2, 40)

    filePath = GetExceedanceFilePath(ModelData_NSSL, windows, start_hour, end_hour)
    params = {"windows": tuple(windows), "start_hour": start_hour, "end_hour": end_hour,
              "thresholds": np.asarray(thresholds), "labels": tuple(labels), "colors": tuple(colors)}

    # ---- load if present and made with the same settings
    if os.path.exists(filePath) and not overwrite:
        with open(filePath, "rb") as f:
            saved = pickle.load(f)
        p = saved["params"]
        same = (p["windows"] == params["windows"] and p["start_hour"] == start_hour
                and p["end_hour"] == end_hour and p["labels"] == params["labels"]
                and p["colors"] == params["colors"]
                and np.array_equal(p["thresholds"], params["thresholds"]))
        if same:
            print(f"Loaded exceedance curves from {filePath}")
            return saved["output"]
        print("Saved settings differ from requested ==> recomputing")

    # ---- otherwise compute and save
    output = BuildExceedanceCurves(ModelData_NSSL, ModelData_TEMPO, windows=windows,
                                   thresholds=thresholds, start_hour=start_hour,
                                   end_hour=end_hour, labels=labels, colors=colors)
    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump({"params": params, "output": output}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved exceedance curves to {filePath}")
    return output

In [ ]:
###################
#PLOTTING FUNCTIONS

In [ ]:
def PlotExceedance(thresholds, curves,
                   markThresholds=(1, 5, 10, 25),
                   legendOrder=("NSSL", "QPESUMS", "TEMPO"),
                   logY=True, axis=None, fig=None,
                   showLegend=True, showXLabel=True, showYLabel=True):

    if axis is None:
        fig, axis = plt.subplots(figsize=(6, 4.5))

    for c in curves:
        axis.plot(thresholds, c["y"], color=c["color"], ls=c.get("ls", "-"),
                  lw=1.8, label=c["label"],
                  zorder=3 if "MRMS" in c["label"] else 2)

    # x label from the window lengths stored with the curves
    windows = sorted({c["window_hours"] for c in curves})
    if len(windows) == 1:
        xlabel = f"{windows[0]}-hourly accumulated rain rate (mm h$^{{-1}}$)"
    else:
        xlabel = f"{'/'.join(str(w) for w in windows)}-hourly accumulated rain rate (mm h$^{{-1}}$)"

    axis.set_xscale("log")
    if logY:
        axis.set_yscale("log")
    if showXLabel: axis.set_xlabel(xlabel, fontsize=fontSettings["labelFont"])
    if showYLabel: axis.set_ylabel("Fraction of grid-points $\\geq$ rate", fontsize=fontSettings["labelFont"])
    for m in markThresholds:
        axis.axvline(m, color="gray", lw=0.5, ls="--")
    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

    handles, labels = axis.get_legend_handles_labels()
    def key(item):
        label = item[1]
        group = next((i for i, g in enumerate(legendOrder) if g in label), len(legendOrder))
        return (group, label)
    handles, labels = zip(*sorted(zip(handles, labels), key=key))
    axis.legend(handles, labels, frameon=False, fontsize=fontSettings["legendFont"])

    if showLegend:
        axis.legend(handles, labels, frameon=False, fontsize=fontSettings["legendFont"], loc='lower left')
    return [fig, axis]

fontSettings = {
    "tickFont": 13.5,
    "labelFont": 14,
    "legendFont": 12,
    "titleFont": 22,
}

In [ ]:
###################
#PLOTTING

In [ ]:
[curves, wetFractions, thr] = LoadOrRun_BuildExceedanceCurves(ModelData_NSSL, ModelData_TEMPO,
                                                              windows=((3, "-"),))
# [fig, _] = PlotExceedance(thr, curves)

In [ ]:
###################
#COMBINE PLOT ALL CASES
plotting=True
# plotting=False #keep false when running the above ones

In [ ]:
#Combined Plotting Functions
def LoadExceedanceFile(Region, Case, spinup_hours, windows=((3, "-"),), start_hour=12, end_hour=None):
    RunType = (Region, Case, "NSSL", spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory,
                                          DirectoryManager.scratchDirectory, RunType, printSummary=False)
    filePath = GetExceedanceFilePath(ModelData, windows, start_hour, end_hour)
    with open(filePath, "rb") as f:
        saved = pickle.load(f)
    return saved["output"]            # [curves, wetFractions, thr]

caseList = [("TRACER", "WET", "0"), ("TRACER", "DIURNAL", "-5"),
            ("PRECIP", "WET", "12"), ("PRECIP", "DIURNAL", "12"),
            ("Hawaii", "WET", "12"), ("Hawaii", "TRADES", "24")]

def PlotExceedanceGrid(caseList, transpose=False, figsize=(11, 12),
                       xlim=(0.1, 100), ylim=(1e-6, 1)):
    regions = ["TRACER", "PRECIP", "Hawaii"]
    nrows, ncols = (2, 3) if transpose else (3, 2)      # default: 3 rows x 2 columns
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=True,
                             gridspec_kw={"wspace": 0.08, "hspace": 0.12})

    for n, (Region, Case, spinup_hours) in enumerate(caseList):
        r = regions.index(Region)
        kind = 0 if Case == "WET" else 1                  # DIURNAL and TRADES go in the second slot
        row, col = (kind, r) if transpose else (r, kind)
        axis = axes[row, col]

        [curves, wetFractions, thr] = LoadExceedanceFile(Region, Case, spinup_hours)
        if Region == "PRECIP":
            for c in curves:
                if c["label"] == "MRMS QPE":
                    c["label"] = "QPESUMS"
        PlotExceedance(thr, curves, axis=axis, fig=fig,
                       showLegend=(row == 0 and col == 0),
                       showXLabel=(row == nrows - 1), showYLabel=(col == 0),
                       legendOrder=("NSSL", "MRMS", "QPESUMS", "TEMPO"))
        axis.set_xlim(*xlim); axis.set_ylim(*ylim)
        axis.set_title(f"{Region} {Case}", fontsize=fontSettings["labelFont"] + 2, fontweight="bold")
    return [fig, axes]

In [ ]:
def SaveFigure(fig, dpi=300, plotType="3hourly", ModelData=None, subDirectory="Combined",
               extension="jpg"):
    """
    Saves an exceedance figure.
      ModelData=None : combined figure, saved in outputPlottingDirectory/<subDirectory>/
      ModelData given: single-case figure, saved in the case's own folder (as in your old function)
    """
    if ModelData is not None:
        outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs"
    else:
        outputSubDirectory = subDirectory
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallExceedance_{plotType}.{extension}"
    )

    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
#Running
if plotting:
    [fig, axes] = PlotExceedanceGrid(caseList)
    SaveFigure(fig, plotType="3hourly")